# MS&E 342 — Full Pipeline on Colab A100

**Reward-Conditioned Reverse Diffusion for Portfolio Optimization**

This notebook:
1. Clones the private GitHub repo
2. Installs all dependencies
3. Patches scripts for CUDA (A100)
4. Runs the full pipeline: data → score model → scenarios → OT → eta sweep → backtest → report
5. Commits all artifacts and pushes back to GitHub

**Runtime:** ~60-90 min on A100 (full 2000-epoch score model, 10k scenarios, 5-eta sweep)

**Required secrets (set in Colab Secrets panel — key icon in left sidebar):**
- `GITHUB_TOKEN` — GitHub personal access token with `repo` scope
- `GITHUB_USERNAME` — your GitHub username (e.g. `siddhant250803`)
- `GITHUB_EMAIL` — your commit email

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch runtime to A100')

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q yfinance cvxpy POT scipy numpy pandas matplotlib

## 2. Authenticate and clone repo

In [ ]:
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = userdata.get('GITHUB_USERNAME')
GITHUB_EMAIL    = userdata.get('GITHUB_EMAIL')
REPO_NAME       = 'mse342-diffusion-portfolio'
REPO_URL        = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
REPO_DIR        = f'/content/{REPO_NAME}'

if os.path.isdir(REPO_DIR):
    print(f"Repo already exists — resetting to origin/main ...")
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'clean', '-fd'], check=True)
    print("Reset complete.")
else:
    print(f"Cloning {GITHUB_USERNAME}/{REPO_NAME} ...")
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print("Clone complete.")

%cd /content/{REPO_NAME}
!git config user.name  "{GITHUB_USERNAME}"
!git config user.email "{GITHUB_EMAIL}"
!git log --oneline -3
print("Ready.")

## 3. Patch scripts: MPS → CUDA

In [ ]:
import glob, re

# Replace all device detection lines with CUDA-first logic
OLD = '"mps" if torch.backends.mps.is_available() else "cpu"'
NEW = '"cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")'

patched = []
for path in glob.glob('experiments/*.py') + ['run_project.py']:
    text = open(path).read()
    if OLD in text:
        open(path, 'w').write(text.replace(OLD, NEW))
        patched.append(path)

print(f"Patched {len(patched)} files: {patched}")

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Active device: {DEVICE}")

## 4. Stage 1 — Data

In [ ]:
!python experiments/01_data.py

## 5. Stage 2 — Train Score Model (2000 epochs, A100)

In [ ]:
import time
t0 = time.time()
!python experiments/02_score_model.py --epochs 2000
print(f"\nScore model trained in {(time.time()-t0)/60:.1f} min")

## 6. Stage 3 — Generate 10,000 Scenarios

In [ ]:
t0 = time.time()
!python experiments/03_sample_and_stylized.py --n_samples 10000
print(f"\nScenario generation: {(time.time()-t0)/60:.1f} min")

## 7. Stage 4 — OT Calibration (Gaussian + Sinkhorn subset)

In [ ]:
t0 = time.time()
!python experiments/07_ot_calibration.py
print(f"\nOT calibration: {(time.time()-t0)/60:.1f} min")

## 8. Stage 5 — OT-Augmented Score Model (optional, ~15 min extra)

In [ ]:
RUN_OT_AUGMENTED = True   # set False to skip

if RUN_OT_AUGMENTED:
    t0 = time.time()
    !python experiments/07_ot_calibration.py --ot-augmented
    print(f"\nOT-augmented model: {(time.time()-t0)/60:.1f} min")
else:
    print("Skipped OT-augmented training.")

## 9. Stage 6 — Eta Sweep on Validation (eta selected on 2021 only)

In [ ]:
t0 = time.time()
!python experiments/08_eta_sweep.py
print(f"\nEta sweep: {(time.time()-t0)/60:.1f} min")

import pandas as pd
print("\nSelected eta:")
print(pd.read_csv('results/eta_selected.csv', index_col=0).to_string())

## 10. Stage 7 — Leakage Audit (must pass)

In [ ]:
import subprocess
result = subprocess.run(['python', 'experiments/00_leakage_audit.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("AUDIT FAILED — fix before proceeding")
    print(result.stderr)
    raise RuntimeError("Leakage audit failed")
else:
    print("AUDIT PASSED — proceeding to final evaluation")

## 11. Stage 8 — Final Backtest (test evaluated once)

In [ ]:
t0 = time.time()
!python experiments/06_compare.py
print(f"\nBacktest: {(time.time()-t0)/60:.1f} min")

print("\n=== FIXED-SCENARIO RESULTS ===")
print(pd.read_csv('results/final_metrics_fixed.csv', index_col=0)
        [['ann_ret','ann_vol','sharpe','sharpe_ci_lo','sharpe_ci_hi','cvar95','max_dd','avg_hhi']]
        .round(4).to_string())

print("\n=== ROLLING RESULTS ===")
print(pd.read_csv('results/final_metrics_rolling.csv', index_col=0)
        [['ann_ret','ann_vol','sharpe','cvar95','max_dd','avg_hhi','turnover']]
        .round(4).to_string())

## 12. Stage 9 — Report Assets

In [ ]:
!python experiments/09_make_final_report_assets.py

print("\n=== FINAL SUMMARY ===")
print(open('results/final_summary.md').read())

## 13. Display Key Figures

In [ ]:
from IPython.display import display, Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

figures = [
    ('figures/stylized_facts_base.png',      'Stylized Facts: Generated vs Historical'),
    ('figures/ot_gaussian_diagnostics.png',  'OT Calibration Diagnostics'),
    ('figures/eta_validation_frontier.png',  'Eta Selection (Validation Only)'),
    ('figures/final_comparison_fixed.png',   'Final Comparison — Fixed Scenario'),
    ('figures/final_comparison_rolling.png', 'Final Comparison — Rolling'),
    ('figures/drawdown_comparison_fixed.png','Drawdown Comparison'),
    ('figures/portfolio_concentration_fixed.png', 'Concentration & Turnover'),
]

for path, title in figures:
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        ax.set_title(title, fontsize=13, pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Missing: {path}')

## 14. Re-run Leakage Audit (final check before push)

In [ ]:
result = subprocess.run(['python', 'experiments/00_leakage_audit.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Final audit failed — not pushing")

## 15. Compile LaTeX Paper

In [ ]:
import shutil, subprocess

# Install pdflatex if not present (~3 min on Colab)
if not shutil.which('pdflatex'):
    print('Installing texlive (this takes ~3 min) ...')
    subprocess.run(['apt-get', 'install', '-y', '-q',
                    'texlive-latex-base', 'texlive-latex-extra',
                    'texlive-fonts-recommended', 'texlive-science'],
                   check=True)
    print('texlive installed.')

latex_result = subprocess.run(
    ['pdflatex', '-interaction=nonstopmode', 'paper.tex'],
    capture_output=True, text=True
)
if 'Output written on' in latex_result.stdout:
    # Run twice for cross-references
    subprocess.run(['pdflatex', '-interaction=nonstopmode', 'paper.tex'],
                   capture_output=True)
    print('paper.pdf compiled successfully')
    pages_line = [l for l in latex_result.stdout.split('
') if 'Output written' in l]
    print(pages_line[0] if pages_line else '')
else:
    print('LaTeX compile failed — check paper.tex')
    print(latex_result.stdout[-3000:])
    print(latex_result.stderr[-1000:])

## 16. Commit and Push All Results to GitHub

In [ ]:
from datetime import datetime

run_ts = datetime.now().strftime('%Y-%m-%d %H:%M UTC')

# Stage all changed/new files
!git add -A

# Show what's changed
status = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True)
print(f"Files to commit ({len(status.stdout.strip().splitlines())} changed):")
print(status.stdout[:2000])

# Commit
commit_msg = f"""results: full production run on A100 — {run_ts}

- Score model: 2000 epochs on 2014-2020 training data
- Scenarios: 10,000 base + Gaussian OT calibrated
- Eta sweep: validation-selected eta on 2021 only
- Leakage audit: PASSED (0 failures)
- Fixed-scenario backtest: Diffusion+FT Sharpe updated
- Rolling backtest: all strategies evaluated
- paper.pdf: recompiled with production figures

Co-Authored-By: Claude Sonnet 4.6 <noreply@anthropic.com>"""

result = subprocess.run(
    ['git', 'commit', '-m', commit_msg],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode not in (0, 1):
    print(result.stderr)

In [ ]:
# Push to GitHub
push_result = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    capture_output=True, text=True
)
print(push_result.stdout)
print(push_result.stderr)
if push_result.returncode == 0:
    print(f"\nSuccessfully pushed to github.com/{GITHUB_USERNAME}/{REPO_NAME}")
else:
    print("Push failed — check token permissions")

## 17. Summary

In [ ]:
import json

audit = json.load(open('results/leakage_audit_result.json'))
fixed = pd.read_csv('results/final_metrics_fixed.csv', index_col=0)
sel   = pd.read_csv('results/eta_selected.csv', index_col=0)

print("=" * 60)
print("FULL PIPELINE COMPLETE")
print("=" * 60)
print(f"Leakage audit: {audit['status']}  (failures={audit['failures']}, warnings={audit['warnings']})")
print(f"Selected eta*: {sel['selected_eta'].values[0]}  "
      f"(val_sharpe={float(sel['validation_score'].values[0]):.3f})")
print()
print("Fixed-scenario test Sharpe (2022-2024):")
for strat in fixed.index:
    s = fixed.loc[strat, 'sharpe']
    lo = fixed.loc[strat, 'sharpe_ci_lo']
    hi = fixed.loc[strat, 'sharpe_ci_hi']
    print(f"  {strat:<38}  {s:+.3f}  95% CI [{lo:+.2f}, {hi:+.2f}]")
print()
print(f"All artifacts committed and pushed to:")
print(f"  https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")